# TiniMind Training v3
LLM Bahasa Indonesia dari nol — pretrain + SFT

**Perubahan terbaru**
```
TiniMind_Prototype/
├── quantize ← diubah bair support sama TiniMind
└── train_tokenizer_indo ← diubah bair support nama TiniMind
```

In [ ]:
# Cell 1 — Mount Drive & Setup
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob

BASE         = '/content/drive/MyDrive/TiniMind'
DATA_DIR     = f'{BASE}/data/mc4_indo'          # chunk_*.bin dari Wikipedia + CulturaX
PRETRAIN_DIR = f'{BASE}/output/pretrain'         # checkpoint pretrain
SFT_DIR      = f'{BASE}/output/sft'              # checkpoint SFT
SFT_DATA_DIR = f'{BASE}/data/sft'               # chunk SFT yang sudah ditokenisasi
SFT_JSONL    = f'{BASE}/data/sft_200_final.jsonl'
TOK_PATH     = f'{BASE}/tokenizer/indo_bpe_32k.model'

# Hanya buat 2 folder output yang diperlukan
os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(SFT_DIR,      exist_ok=True)

sys.path.insert(0, BASE)

import torch
print(f'Device : {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Mounted at /content/drive
Device : cpu


In [ ]:
# Cell 2 — Install dependencies
%pip install -q sentencepiece datasets

In [ ]:
# Cell 3 — Cek data & checkpoint pretrain terakhir
chunks    = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
total_tok = sum(os.path.getsize(f)//2 for f in chunks)
print(f'Chunks    : {len(chunks)} file')
print(f'Total tok : {total_tok/1e9:.2f}B')

# Checkpoint pretrain — selalu cari di output/pretrain/
pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
if pretrain_ckpts:
    RESUME = pretrain_ckpts[-1]
    print(f'Resume    : {os.path.basename(RESUME)}')
else:
    RESUME = None
    print('Resume    : tidak ada — training dari awal')

if not chunks:
    print('\n⚠️  Belum ada data! Jalankan Cell 4 dulu.')

Chunks    : 301 file
Total tok : 3.00B
Resume    : step_0007000_loss_3.3800.pt


In [ ]:
# Cell 4 — Stream & tokenize CulturaX → chunk_*.bin
# SKIP kalau data/mc4_indo sudah punya cukup chunk (3B token = ~300 file)

import sentencepiece as spm
import numpy as np
from datasets import load_dataset

TARGET_TOKENS = 3_000_000_000
CHUNK_SIZE    = 10_000_000

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)
print(f'Tokenizer vocab: {sp.GetPieceSize()}')

os.makedirs(DATA_DIR, exist_ok=True)
existing   = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
tokens_done = sum(os.path.getsize(f)//2 for f in existing)
print(f'Token sudah ada: {tokens_done/1e9:.2f}B / {TARGET_TOKENS/1e9:.1f}B')

if tokens_done >= TARGET_TOKENS:
    print('Target sudah tercapai, skip.')
else:
    chunk_idx = len(existing)
    buf = []
    ds  = load_dataset('uonlp/CulturaX', 'id', split='train', streaming=True, trust_remote_code=True)

    for doc in ds:
        toks = sp.Encode(doc['text'])
        buf.extend(toks)
        tokens_done += len(toks)

        while len(buf) >= CHUNK_SIZE:
            path = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
            np.array(buf[:CHUNK_SIZE], dtype=np.uint16).tofile(path)
            print(f'Saved chunk_{chunk_idx:04d}.bin | total: {tokens_done/1e9:.2f}B')
            buf = buf[CHUNK_SIZE:]
            chunk_idx += 1

        if tokens_done >= TARGET_TOKENS:
            print('Target tercapai!')
            break

    if buf:
        path = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
        np.array(buf, dtype=np.uint16).tofile(path)
        print(f'Final: {os.path.basename(path)}')

In [ ]:
# Cell 5 — Pretrain (auto-remount kalau Drive disconnect)
import subprocess, time, glob, os

def remount_drive():
    from google.colab import drive
    subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(3)
    print('Drive remounted!')

def get_latest_checkpoint():
    ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
    return ckpts[-1] if ckpts else None

def run_training():
    ckpt = get_latest_checkpoint()
    resume_arg = f'--resume {ckpt}' if ckpt else ''
    cmd = (
        f'python {BASE}/train.py'
        f' --config prod_300m'
        f' --data-dir {DATA_DIR}'
        f' --output-dir {PRETRAIN_DIR}'
        f' --dtype fp16'
        f' --max-steps 20000'
        f' --lr 3e-4'
        f' --batch-size 4'
        f' --grad-accum 8'
        f' --seq-len 1024'
        f' --log-every 100'
        f' --save-every 1000'
        f' --eval-every 1000'
        f' {resume_arg}'
    )
    if ckpt:
        print(f'Resume dari: {os.path.basename(ckpt)}')
    return subprocess.run(cmd, shell=True).returncode

# Loop otomatis — kalau Drive disconnect, remount dan lanjut dari checkpoint terakhir
MAX_RETRIES = 20
for attempt in range(MAX_RETRIES):
    print(f'--- Training attempt {attempt+1} ---')
    returncode = run_training()
    if returncode == 0:
        print('Training selesai!')
        break
    print(f'Training berhenti (code={returncode}). Remounting Drive...')
    try:
        remount_drive()
    except Exception as e:
        print(f'Remount error: {e}, tunggu 10 detik...')
        time.sleep(10)


--- Training attempt 1 ---
Resume dari: step_0007000_loss_3.3800.pt
Training berhenti (code=1). Remounting Drive...
Mounted at /content/drive
Drive remounted!
--- Training attempt 2 ---
Resume dari: step_0007000_loss_3.3800.pt


In [ ]:
# Cell 6 — Tokenize SFT data
# Support format campuran: compact JSONL (1 baris) + pretty-printed (multi-baris)
# Format input: {"turns": [["user", "..."], ["assistant", "..."]]}

import sentencepiece as spm, json, numpy as np

os.makedirs(SFT_DATA_DIR, exist_ok=True)

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

# Parse mixed format (compact + pretty-printed JSON objects)
raw     = open(SFT_JSONL).read()
decoder = json.JSONDecoder()
entries, i = [], 0
while i < len(raw):
    while i < len(raw) and raw[i] in ' \t\n\r': i += 1
    if i >= len(raw): break
    try:
        obj, end = decoder.raw_decode(raw, i)
        entries.append(obj)
        i = end
    except json.JSONDecodeError:
        nl = raw.find('\n', i)
        i  = nl + 1 if nl != -1 else len(raw)

print(f'SFT entries parsed: {len(entries)}')

all_tokens = []
for conv in entries:
    text = ''
    for role, content in conv['turns']:
        if role == 'user':
            text += f'<penggunna>{content}</penggunna>'
        else:
            text += f'<asisten>{content}</asisten>'
    all_tokens.extend(sp.Encode(text))

out_path = f'{SFT_DATA_DIR}/chunk_0000.bin'
np.array(all_tokens, dtype=np.uint16).tofile(out_path)
print(f'SFT tokens : {len(all_tokens):,}')
print(f'Saved      : {out_path}')


In [ ]:
# Cell 7 — SFT
# Checkpoint disimpan di: output/sft/step_XXXXXXX_loss_X.XXXX.pt
# Mulai dari checkpoint pretrain terbaik

pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
if not pretrain_ckpts:
    raise RuntimeError('Tidak ada checkpoint pretrain! Jalankan Cell 5 dulu.')

BEST_PRETRAIN = pretrain_ckpts[-1]
print(f'Base model: {os.path.basename(BEST_PRETRAIN)}')

# Cek resume SFT
sft_ckpts  = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
SFT_RESUME = f'--resume {sft_ckpts[-1]}' if sft_ckpts else f'--resume {BEST_PRETRAIN}'

!python {BASE}/train.py \
    --config prod_300m \
    --data-dir {SFT_DATA_DIR} \
    --output-dir {SFT_DIR} \
    --dtype fp16 \
    --max-steps 500 \
    --lr 1e-5 \
    --batch-size 2 \
    --grad-accum 4 \
    --seq-len 1024 \
    --log-every 50 \
    --save-every 100 \
    --eval-every 100 \
    --val-chunks 0 \
    {SFT_RESUME}

In [ ]:
# Cell 8 (UPDATED) — Inference test dengan fix unicode/byte-fallback
#
# GANTI seluruh isi Cell 8 lama dengan kode di bawah ini.
# Perbedaan dari versi lama: pakai generate.py yang sudah fix masalah
# U+FFFD (karakter kotak) yang muncul akibat byte-fallback token yang
# tidak lengkap saat di-decode.

import sys, glob, os
sys.path.insert(0, BASE)

from generate import load_model_and_tokenizer, generate_with_maturity_warning

# Cari checkpoint terbaik: SFT dulu, fallback ke pretrain
sft_ckpts      = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))
ckpt_path      = (sft_ckpts or pretrain_ckpts)[-1]
print(f'Load: {os.path.basename(ckpt_path)}')

model, sp, device = load_model_and_tokenizer(ckpt_path, TOK_PATH)

import torch
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
step = ckpt.get('step', None)

# Contoh pemakaian — generate_with_maturity_warning otomatis format
# prompt dengan <penggunna>...</penggunna><asisten> dan kasih warning
# kalau checkpoint masih terlalu awal untuk hasil koheren.
result = generate_with_maturity_warning(
    model, sp,
    prompt_raw="Apa itu kecerdasan buatan?",
    device=device,
    step=step,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
)
print()
print(result)

Load: step_0010000_loss_3.3357.pt
Flash Attention aktif (F.scaled_dot_product_attention) | vocab=32000
Params: 270.6M
<penggunna>Apa itu kecerdasan buatan?</penggunna><asisten> adalahudah��gabung�apatanggapapat_udah�apat_�udah����usah�ustambang�ucugaugagabungikutirafuga��enggah����arif��ip��enggipungkapetikapat����ungkap�ida�rafarifertiikuti���_�udahudah���ungkapea�gabung��rans�engggabambi��apatikuti����_��enggengg�rafanggapanustubah�apatarisan�uga����ubahenggubahichoke���_gabikuti��ea��apat�raf�ikuti_ida�enggokeapat���engg�ertieaapat�ongkubahanggapanip��gabungidaarifanggapanggap�anggapan�ugarafudah��erti��itung�usahego��idaugaugaustusah��usahakanapatitik����usahistr�okeistress��uga�raf


In [ ]:
# Cell 9 — Quantize INT8
sft_ckpts  = sorted(glob.glob(f'{SFT_DIR}/step_*.pt'))
ckpt_path  = sft_ckpts[-1] if sft_ckpts else sorted(glob.glob(f'{PRETRAIN_DIR}/step_*.pt'))[-1]
quant_out  = f'{BASE}/output/tinimind_int8.pt'

!python {BASE}/quantize.py \
    --checkpoint {ckpt_path} \
    --output {quant_out} \
    --mode dynamic